## Preprocesamiento de la data

### Carga de las librerías y de los módulos

Se cargan las librerías para el manejo de rutas, variables numéricas y configuración del entorno de ejecución.

Por lo que se agrega el directorio padre al path para poder importar los módulos desarrollados dentro de la carpeta `src`.
Se importan los módulos del proyecto que contienen las funciones de carga, transformación y feature engineering, empleando la función `reload` de `importlib`, con el fin de ir actualizando las dunciones dentro del notebook sin reiniciar todo el entorno.

In [1]:
from pathlib import Path

RAIZ_PROYECTO = "."

In [2]:
RAIZ_PROYECTO = Path(RAIZ_PROYECTO)

print(f"Raíz del proyecto: {RAIZ_PROYECTO}")

Raíz del proyecto: .


In [3]:
from pathlib import Path
import sys

# Convertir la ruta recibida a un objeto Path
RAIZ_PROYECTO = Path(RAIZ_PROYECTO)

# Si el notebook se ejecuta directamente desde la carpeta notebooks,
# la raíz del proyecto corresponde al directorio padre.
if str(RAIZ_PROYECTO) == ".":
    RAIZ_PROYECTO = Path.cwd().parent
else:
    RAIZ_PROYECTO = RAIZ_PROYECTO.resolve()

# Agregar la raíz del proyecto al buscador de módulos de Python
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

In [4]:
import pandas as pd
import importlib

import json
from datetime import datetime

from src import load
from src import feature_engineering

In [40]:
importlib.reload(load)
importlib.reload(feature_engineering)

<module 'src.feature_engineering' from 'd:\\OneDrive - SUPERINTENDENCIA DE BANCOS\\Escritorio\\Proyecto_usfq\\src\\feature_engineering.py'>

### Integración de los datos de todos los indicadores

En esta etapa se integran en un único dataset de análisis los indicadores financieros extraídos de los boletines de la Superintendencia de Bancos, los indicadores construidos y derivados de la cartera y las variaciones construidas de las principales cuentas del balance,por medio de un pipeline elaborado con las funciones de los scripts cargadas de la carpeta `src`.


Para este fin, el propio ente de control advierte en su página de boletines financieros mensuales (https://www.superbancos.gob.ec/estadisticas/portalestudios/bancos/) que: "....Los boletines financieros mensuales de Sistema Financiero se prepararon conforme los planes y catálogos de cuentas vigentes a la fecha de corte del boletín. En los datos estadísticos de los boletines y los boletines en Serie se realizó un proceso de homologación de la información financiera para que los datos sean comparables en el tiempo (reexpresar los datos históricos a un plan de cuentas, en este caso se han homologado a la normativa vigente desde 1 mayo 2021)."

Este detalle de fechas es importante puesto que, para evitar problemas de unficación de toda la data es necesario que los excels estén homologados. Para el presente proyecto se consideran los boletines financieros mensuales desde Mayo de 2021. Al incluir las variaciones interanuales de las cuentas principales del balance nuestro análisis se rezaga a un año después, por lo que nuestro dataset comienza desde Mayo 2022 hasta el último boletín publicado por el ente de control de Julio 2026.


La integración se realiza utilizando como claves de identificación el año, mes y entidad financiera.

In [6]:
anios=range(2022,2027)

In [7]:
# Lista donde se almacenará la información integrada de cada año.
datasets_anuales=[]
# Se recorre para cada año el pipeline de preprocessing y feature engineering para los indicadores generales financieros y los indicadores de cartera
#  y se almacena los resultados por año para posteriormente
# integrar todos los años en una sola base
for i in anios:
    indicadores=load.procesar_indicadores(i, load.archivos,load.meses, load.columnas_indicadores)
    lista_cartera=load.procesar_cartera(i, load.archivos,load.meses)
    indicadores_cartera=feature_engineering.feature_engineering_cartera(lista_cartera)
    lista_indicadores_y_cartera=feature_engineering.unir_todos_indicadores(indicadores, indicadores_cartera)
    indicadores_y_cartera=pd.concat(lista_indicadores_y_cartera,
        ignore_index=True)
    datasets_anuales.append(indicadores_y_cartera)

Leyendo archivo: d:\OneDrive - SUPERINTENDENCIA DE BANCOS\Escritorio\Proyecto_usfq\data\2022\FINANCIERO MENSUAL BANCA PRIVADA 2022_01.xlsx
Leyendo archivo: d:\OneDrive - SUPERINTENDENCIA DE BANCOS\Escritorio\Proyecto_usfq\data\2022\FINANCIERO MENSUAL BANCA PRIVADA 2022_02.xlsx
Leyendo archivo: d:\OneDrive - SUPERINTENDENCIA DE BANCOS\Escritorio\Proyecto_usfq\data\2022\FINANCIERO MENSUAL BANCA PRIVADA 2022_03.xlsx
Leyendo archivo: d:\OneDrive - SUPERINTENDENCIA DE BANCOS\Escritorio\Proyecto_usfq\data\2022\FINANCIERO MENSUAL BANCA PRIVADA 2022_04.xlsx
Leyendo archivo: d:\OneDrive - SUPERINTENDENCIA DE BANCOS\Escritorio\Proyecto_usfq\data\2022\FINANCIERO MENSUAL BANCA PRIVADA 2022_05_rev.xlsx
Leyendo archivo: d:\OneDrive - SUPERINTENDENCIA DE BANCOS\Escritorio\Proyecto_usfq\data\2022\FINANCIERO MENSUAL BANCA PRIVADA 2022_06.xlsx
Leyendo archivo: d:\OneDrive - SUPERINTENDENCIA DE BANCOS\Escritorio\Proyecto_usfq\data\2022\FINANCIERO MENSUAL BANCA PRIVADA 2022_07.xlsx
Leyendo archivo: d:\One

In [8]:
# Se concatenan verticalmente los datasets correspondientes
# a cada año para obtener una única base histórica.
#
# ignore_index=True permite generar un nuevo índice consecutivo
# para el DataFrame resultante.

dataset_indicadores_cartera = pd.concat(
    datasets_anuales,
    ignore_index=True
)

In [9]:
dataset_indicadores_cartera

,MES,AÑO,ENTIDAD,( PATRIMONIO + RESULTADOS ) / ACTIVOS INMOVILIZADOS NETOS (3) (6),INDICE DE CAPITALIZACION NETO: FK / FI,ACTIVOS IMPRODUCTIVOS NETOS / TOTAL ACTIVOS,ACTIVOS PRODUCTIVOS / TOTAL ACTIVOS,ACTIVOS PRODUCTIVOS / PASIVOS CON COSTO,MOROSIDAD DE LA CARTERA DE CREDITOS CONSUMO,MOROSIDAD DE LA CARTERA DE CRÉDITOS INMOBILIARIO,...,GASTOS DE PERSONAL ESTIMADOS / ACTIVO PROMEDIO (3),RESULTADOS DEL EJERCICIO / PATRIMONIO PROMEDIO,RESULTADOS DEL EJERCICIO / ACTIVO PROMEDIO,CARTERA BRUTA / (DEPOSITOS A LA VISTA + DEPOSITOS A PLAZO),FONDOS DISPONIBLES / TOTAL DEPOSITOS A CORTO PLAZO,CARTERA IMPRODUCTIVA / CARTERA BRUTA,CARTERA VENCIDA / CARTERA BRUTA,CARTERA QUE NO DEVENGA INTERESES / CARTERA BRUTA,CARTERA REFINANCIADA / CARTERA BRUTA,CARTERA REESTRUCTURADA / CARTERA BRUTA
0,Enero,2022,BP GUAYAQUIL,4.141498,0.084334,0.14544,0.85456,1.20628,0.01425,0.022054,...,0.015297,0.128519,0.012238,0.880435,0.21366,0.012428,0.005401,0.007027,0.015168,0.004739
1,Enero,2022,BP PACIFICO,2.6451,0.097085,0.223376,0.77662,1.10437,0.032261,0.034234,...,0.009043,0.102842,0.012095,0.854694,0.341741,0.031243,0.015333,0.015910,0.109175,0.039099
2,Enero,2022,BP PICHINCHA,-23.726736,0.092786,0.093155,0.90684,1.43429,0.032347,0.045745,...,0.011048,0.113642,0.011441,0.830851,0.178394,0.028914,0.007745,0.021169,0.029904,0.055667
3,Enero,2022,BP PRODUBANCO,3.21609,0.0715,0.144305,0.85569,1.43589,0.032538,0.032279,...,0.010637,0.127418,0.010326,0.86819,0.320233,0.015483,0.003208,0.012275,0.007283,0.026373
4,Enero,2022,BP AUSTRO,1.391251,0.078349,0.167015,0.83298,1.05067,0.073443,0.016077,...,0.010616,0.035847,0.003303,0.671742,0.250919,0.039514,0.014388,0.025126,0.079909,0.026752
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1299,Julio,2026,BP CAPITAL,0.743608,0.088779,0.203118,0.796882,0.974991,0.035269,0,...,0.022017,0.000602,0.00008,0.762553,0.241408,0.043296,0.002111,0.041185,0.014315,0.002475
1300,Julio,2026,BP DELBANK,7.435204,0.288765,0.157979,0.842021,1.769811,0.105913,0.000002,...,0.040853,-0.003508,-0.001117,1.275196,0.591112,0.031132,0.010960,0.020171,0.068912,0.003849
1301,Julio,2026,BP D-MIRO S.A./BANCO ATLÁNTIDA S.A.,0.788369,0.055191,0.128191,0.871809,0.96599,0.000358,0,...,0.030823,0.00906,0.000702,0.535486,0.331037,0.021523,0.009165,0.012358,0.028839,0.007510
1302,Julio,2026,"BP BANCO DESARROLLO DE LOS PUEBLOS S.A., COD...",2.990621,0.103658,0.105249,0.894751,1.069191,0.079619,0.006083,...,0.021339,0.024556,0.0029,0.954322,0.17721,0.099170,0.040266,0.058904,0.041134,0.006622


In [10]:
# Se realiza una revisión puntual de una entidad específica
# para verificar la consistencia de sus registros a través
# de los diferentes períodos.

dataset_indicadores_cartera["ENTIDAD"].unique()

array(['BP GUAYAQUIL', 'BP PACIFICO', 'BP PICHINCHA', 'BP PRODUBANCO',
       'BP AUSTRO', 'BP BOLIVARIANO', 'BP CITIBANK', 'BP DINERS',
       'BP GENERAL RUMIÑAHUI', 'BP INTERNACIONAL', 'BP LOJA',
       'BP MACHALA', 'BP SOLIDARIO', 'BP PROCREDIT', 'BP AMAZONAS',
       'BP BANCO COMERCIAL DE MANABI', 'BP LITORAL', 'BP COOPNACIONAL',
       'BP CAPITAL', 'BP FINCA S.A./BANCO AMIBANK S.A.', 'BP DELBANK',
       'BP D-MIRO S.A./BANCO ATLÁNTIDA S.A.',
       'BP BANCO  DESARROLLO DE LOS PUEBLOS  S.A., CODESARROLLO',
       'BP VISIONFUND ECUADOR S.A.'], dtype=object)

In [11]:
# Ahora se carga la lista que incluye las principales cuentas de los balances contables financieros
# y con la función de feature_engineering se construyen las nuevas variables con las variaciones interanuales 

lista_balance=load.procesar_balances(load.archivos,load.meses, load.cuentas)
indicadores_balance=feature_engineering.calcular_variaciones_balance(lista_balance)

In [12]:
lista_balance[25].info()

<class 'pandas.DataFrame'>
Index: 24 entries, 0 to 25
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   MES                   24 non-null     str    
 1   AÑO                   24 non-null     int64  
 2   ENTIDAD               24 non-null     str    
 3   FONDOS DISPONIBLES    24 non-null     float64
 4   INVERSIONES           24 non-null     float64
 5   CARTERA DE CRÉDITOS   24 non-null     float64
 6   TOTAL ACTIVO          24 non-null     float64
 7   Depósitos a la vista  24 non-null     float64
 8   Depósitos a plazo     24 non-null     float64
 9   TOTAL PATRIMONIO      24 non-null     float64
dtypes: float64(7), int64(1), str(2)
memory usage: 2.1 KB


In [13]:
indicadores_balance.columns.to_list()

['MES',
 'AÑO',
 'ENTIDAD',
 'VAR_FONDOS DISPONIBLES',
 'VAR_INVERSIONES',
 'VAR_CARTERA DE CRÉDITOS',
 'VAR_TOTAL ACTIVO',
 'VAR_Depósitos a la vista',
 'VAR_Depósitos a plazo',
 'VAR_TOTAL PATRIMONIO']

In [14]:
indicadores_balance

,MES,AÑO,ENTIDAD,VAR_FONDOS DISPONIBLES,VAR_INVERSIONES,VAR_CARTERA DE CRÉDITOS,VAR_TOTAL ACTIVO,VAR_Depósitos a la vista,VAR_Depósitos a plazo,VAR_TOTAL PATRIMONIO
0,Mayo,2022,BP GUAYAQUIL,-0.153575,-0.031059,0.219636,0.108500,0.144194,0.163349,0.080876
1,Mayo,2022,BP PACIFICO,-0.161024,0.152143,-0.006714,-0.002157,-0.037496,0.076166,0.020035
2,Mayo,2022,BP PICHINCHA,-0.345962,0.008894,0.341896,0.106841,0.090957,0.176605,0.043768
3,Mayo,2022,BP PRODUBANCO,-0.045015,-0.035218,0.241502,0.138065,0.092976,0.119563,0.077973
4,Mayo,2022,BP AUSTRO,-0.159378,0.069119,0.214098,0.114007,0.078231,0.048206,0.128133
...,...,...,...,...,...,...,...,...,...,...
1203,Julio,2026,BP CAPITAL,-0.017643,0.545330,0.434054,0.322436,0.046285,0.445646,0.040729
1204,Julio,2026,BP DELBANK,0.096224,-0.379255,-0.118974,-0.102445,-0.125425,-0.227068,0.051840
1205,Julio,2026,BP D-MIRO S.A./BANCO ATLÁNTIDA S.A.,1.398896,0.509052,0.781038,0.712938,0.275121,1.096880,-0.056485
1206,Julio,2026,"BP BANCO DESARROLLO DE LOS PUEBLOS S.A., COD...",-0.585971,0.248656,0.036946,-0.051873,-0.138384,0.086575,0.067031


In [15]:
# Se integran los indicadores generales y de cartera con las  variaciones interanuales del balance.
#
# AÑO, MES y ENTIDAD constituyen la llave lógica que identifica
# cada observación.
#
# Se utiliza un left join para conservar todas las observaciones
# de la base principal y agregar las variaciones del balance
# cuando existe una coincidencia.

dataset_final = dataset_indicadores_cartera.merge(
    indicadores_balance,
    on=["MES", "AÑO", "ENTIDAD"],
    how="left"
)

### Validación y preparación del dataset

En esta etapa se verifica la estructura, dimensionalidad, tipos de datos, valores nulos, duplicados y valores infinitos de la base integrada.

El objetivo es asegurar que la información tenga una estructura consistente antes de utilizarla en las etapas posteriores de análisis exploratorio y modelamiento.

In [16]:
# Se verifica la cantidad inicial de filas y columnas de la base.
dataset_final.shape

(1304, 34)

In [17]:
# Se excluyen los registros correspondientes a enero-abril de 2022.
#
# Estos períodos no serán considerados en el análisis debido a que
# no forman parte del período comparable definido para el estudio y especificado por el ente de control

dataset_final = dataset_final[
    ~(
        (dataset_final["AÑO"] == 2022) &
        (dataset_final["MES"].isin(["Enero", "Febrero", "Marzo", "Abril"]))
    )
].copy()

In [18]:
# Se revisa la estructura general del DataFrame:
# número de observaciones, columnas, tipos de datos y valores no nulos.
dataset_final.shape

(1208, 34)

In [19]:
# Se revisan específicamente los tipos de datos de las variables
# posteriores a las tres primeras columnas de identificación.
dataset_final

,MES,AÑO,ENTIDAD,( PATRIMONIO + RESULTADOS ) / ACTIVOS INMOVILIZADOS NETOS (3) (6),INDICE DE CAPITALIZACION NETO: FK / FI,ACTIVOS IMPRODUCTIVOS NETOS / TOTAL ACTIVOS,ACTIVOS PRODUCTIVOS / TOTAL ACTIVOS,ACTIVOS PRODUCTIVOS / PASIVOS CON COSTO,MOROSIDAD DE LA CARTERA DE CREDITOS CONSUMO,MOROSIDAD DE LA CARTERA DE CRÉDITOS INMOBILIARIO,...,CARTERA QUE NO DEVENGA INTERESES / CARTERA BRUTA,CARTERA REFINANCIADA / CARTERA BRUTA,CARTERA REESTRUCTURADA / CARTERA BRUTA,VAR_FONDOS DISPONIBLES,VAR_INVERSIONES,VAR_CARTERA DE CRÉDITOS,VAR_TOTAL ACTIVO,VAR_Depósitos a la vista,VAR_Depósitos a plazo,VAR_TOTAL PATRIMONIO
96,Mayo,2022,BP GUAYAQUIL,4.106345,0.081208,0.133465,0.866535,1.204264,0.015497,0.018206,...,0.007049,0.017101,0.005272,-0.153575,-0.031059,0.219636,0.108500,0.144194,0.163349,0.080876
97,Mayo,2022,BP PACIFICO,2.76007,0.095491,0.217156,0.782844,1.112745,0.032618,0.044019,...,0.017987,0.095438,0.041668,-0.161024,0.152143,-0.006714,-0.002157,-0.037496,0.076166,0.020035
98,Mayo,2022,BP PICHINCHA,-12.882069,0.087281,0.087276,0.912724,1.423879,0.02102,0.042779,...,0.017432,0.021698,0.047222,-0.345962,0.008894,0.341896,0.106841,0.090957,0.176605,0.043768
99,Mayo,2022,BP PRODUBANCO,3.457887,0.07046,0.119412,0.880588,1.46171,0.029096,0.02892,...,0.010628,0.006073,0.024845,-0.045015,-0.035218,0.241502,0.138065,0.092976,0.119563,0.077973
100,Mayo,2022,BP AUSTRO,1.295431,0.077153,0.160334,0.839666,1.042581,0.077468,0.016729,...,0.025212,0.070088,0.027836,-0.159378,0.069119,0.214098,0.114007,0.078231,0.048206,0.128133
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1299,Julio,2026,BP CAPITAL,0.743608,0.088779,0.203118,0.796882,0.974991,0.035269,0,...,0.041185,0.014315,0.002475,-0.017643,0.545330,0.434054,0.322436,0.046285,0.445646,0.040729
1300,Julio,2026,BP DELBANK,7.435204,0.288765,0.157979,0.842021,1.769811,0.105913,0.000002,...,0.020171,0.068912,0.003849,0.096224,-0.379255,-0.118974,-0.102445,-0.125425,-0.227068,0.051840
1301,Julio,2026,BP D-MIRO S.A./BANCO ATLÁNTIDA S.A.,0.788369,0.055191,0.128191,0.871809,0.96599,0.000358,0,...,0.012358,0.028839,0.007510,1.398896,0.509052,0.781038,0.712938,0.275121,1.096880,-0.056485
1302,Julio,2026,"BP BANCO DESARROLLO DE LOS PUEBLOS S.A., COD...",2.990621,0.103658,0.105249,0.894751,1.069191,0.079619,0.006083,...,0.058904,0.041134,0.006622,-0.585971,0.248656,0.036946,-0.051873,-0.138384,0.086575,0.067031


In [20]:
dataset_final.iloc[:, 3:].dtypes

( PATRIMONIO + RESULTADOS ) / ACTIVOS INMOVILIZADOS NETOS (3) (6)     object
INDICE DE CAPITALIZACION NETO: FK / FI                                object
ACTIVOS IMPRODUCTIVOS NETOS / TOTAL ACTIVOS                           object
ACTIVOS PRODUCTIVOS / TOTAL ACTIVOS                                   object
ACTIVOS PRODUCTIVOS / PASIVOS CON COSTO                               object
MOROSIDAD DE LA CARTERA DE CREDITOS CONSUMO                           object
MOROSIDAD DE LA CARTERA DE CRÉDITOS INMOBILIARIO                      object
MOROSIDAD DE LA CARTERA DE CRÉDITOS MICROCRÉDITO                      object
MOROSIDAD DE LA CARTERA TOTAL                                         object
COBERTURA DE LA CARTERA REFINANCIADA                                  object
COBERTURA DE LA CARTERA REESTRUCTURADA                                object
COBERTURA DE LA CARTERA PROBLEMÁTICA                                  object
GASTOS DE OPERACION ESTIMADOS / TOTAL ACTIVO PROMEDIO (3)             object

In [21]:
# Se separan las variables de identificación de las variables
# utilizadas como indicadores financieros.
#
# MES, AÑO y ENTIDAD identifican cada observación, mientras que
# las demás columnas corresponden a variables cuantitativas.

columnas_numericas = [
    col for col in dataset_final.columns
    if col not in ["MES", "AÑO", "ENTIDAD"]
]

columnas_no_numericas = [ "MES", "AÑO", "ENTIDAD"]


In [22]:
# Se convierten las variables financieras a valores numéricos.
#
# errors="coerce" transforma en NaN aquellos valores que no puedan
# convertirse correctamente a número, evitando que un valor inválido
# detenga todo el procesamiento.
#
# Finalmente, se utiliza float64 para mantener un tipo numérico
# homogéneo y compatible con los análisis posteriores.

dataset_final[columnas_numericas] = (
    dataset_final[columnas_numericas]
    .apply(pd.to_numeric, errors="coerce")
    .astype("float64")
)

In [23]:
# Se convierten las variables de identificación a texto para
# mantener un formato homogéneo en la base

dataset_final[columnas_no_numericas] = (
    dataset_final[columnas_no_numericas]
    .astype(str)
)

In [24]:
dataset_final.describe()

,( PATRIMONIO + RESULTADOS ) / ACTIVOS INMOVILIZADOS NETOS (3) (6),INDICE DE CAPITALIZACION NETO: FK / FI,ACTIVOS IMPRODUCTIVOS NETOS / TOTAL ACTIVOS,ACTIVOS PRODUCTIVOS / TOTAL ACTIVOS,ACTIVOS PRODUCTIVOS / PASIVOS CON COSTO,MOROSIDAD DE LA CARTERA DE CREDITOS CONSUMO,MOROSIDAD DE LA CARTERA DE CRÉDITOS INMOBILIARIO,MOROSIDAD DE LA CARTERA DE CRÉDITOS MICROCRÉDITO,MOROSIDAD DE LA CARTERA TOTAL,COBERTURA DE LA CARTERA REFINANCIADA,...,CARTERA QUE NO DEVENGA INTERESES / CARTERA BRUTA,CARTERA REFINANCIADA / CARTERA BRUTA,CARTERA REESTRUCTURADA / CARTERA BRUTA,VAR_FONDOS DISPONIBLES,VAR_INVERSIONES,VAR_CARTERA DE CRÉDITOS,VAR_TOTAL ACTIVO,VAR_Depósitos a la vista,VAR_Depósitos a plazo,VAR_TOTAL PATRIMONIO
count,1208.000000,1208.000000,1208.000000,1208.000000,1208.000000,1208.000000,1208.000000,1208.000000,1208.000000,1208.000000,...,1207.000000,1207.000000,1207.000000,1208.000000,1208.000000,1208.000000,1208.000000,1208.000000,1208.000000,1208.000000
mean,16.564382,0.113417,0.129656,0.870344,1.452182,0.049878,0.066555,0.089563,0.047017,1.040733,...,0.031721,0.036880,0.019650,0.114038,0.229692,0.099536,0.097517,0.190345,25.440645,0.058678
std,508.772993,0.096263,0.067545,0.067545,1.548236,0.039261,0.207324,0.125578,0.046990,1.290756,...,0.030505,0.035211,0.020487,0.452901,0.599447,0.194632,0.133781,0.655566,437.046134,0.135612
min,-1123.052066,-2.589472,-0.002645,0.583179,0.212959,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,-0.731714,-0.761311,-1.000000,-0.884704,-0.684559,-1.000000,-0.772532
25%,0.884036,0.079909,0.085705,0.841721,1.059337,0.028479,0.000000,0.024433,0.023536,0.584152,...,0.016404,0.009921,0.004368,-0.124675,-0.086429,0.034203,0.029916,-0.030455,0.030487,0.013970
50%,2.636429,0.092059,0.115675,0.884325,1.140050,0.046086,0.014703,0.052714,0.034540,0.814337,...,0.024161,0.026232,0.013293,0.041677,0.137785,0.098010,0.091172,0.070587,0.122532,0.063610
75%,5.604946,0.142611,0.158279,0.914295,1.306533,0.062934,0.028413,0.112452,0.054207,1.058919,...,0.035671,0.054642,0.027514,0.236876,0.368646,0.155629,0.143349,0.224256,0.231238,0.103479
max,17606.001928,0.341963,0.416821,1.002645,23.509207,0.430763,1.000000,1.000000,0.313073,14.657317,...,0.205025,0.189812,0.116167,7.810275,4.724520,1.067010,0.849168,8.649531,9494.059333,1.048895


Se multiplican las variables numéricas por 100 para llevar las proporciones expresadas en formato decimal a una escala porcentual.

In [25]:

# Los indicadores financieros se expresan en proporciones en la
# fuente de cálculo. Se multiplican por 100 para expresarlos
# porcentualmente y facilitar su interpretación.

dataset_final[columnas_numericas] = (
    dataset_final[columnas_numericas] * 100
)

In [26]:
dataset_final.describe()

,( PATRIMONIO + RESULTADOS ) / ACTIVOS INMOVILIZADOS NETOS (3) (6),INDICE DE CAPITALIZACION NETO: FK / FI,ACTIVOS IMPRODUCTIVOS NETOS / TOTAL ACTIVOS,ACTIVOS PRODUCTIVOS / TOTAL ACTIVOS,ACTIVOS PRODUCTIVOS / PASIVOS CON COSTO,MOROSIDAD DE LA CARTERA DE CREDITOS CONSUMO,MOROSIDAD DE LA CARTERA DE CRÉDITOS INMOBILIARIO,MOROSIDAD DE LA CARTERA DE CRÉDITOS MICROCRÉDITO,MOROSIDAD DE LA CARTERA TOTAL,COBERTURA DE LA CARTERA REFINANCIADA,...,CARTERA QUE NO DEVENGA INTERESES / CARTERA BRUTA,CARTERA REFINANCIADA / CARTERA BRUTA,CARTERA REESTRUCTURADA / CARTERA BRUTA,VAR_FONDOS DISPONIBLES,VAR_INVERSIONES,VAR_CARTERA DE CRÉDITOS,VAR_TOTAL ACTIVO,VAR_Depósitos a la vista,VAR_Depósitos a plazo,VAR_TOTAL PATRIMONIO
count,1.208000e+03,1208.000000,1208.000000,1208.000000,1208.000000,1208.000000,1208.000000,1208.000000,1208.000000,1208.000000,...,1207.000000,1207.000000,1207.000000,1208.000000,1208.000000,1208.000000,1208.000000,1208.000000,1208.000000,1208.000000
mean,1.656438e+03,11.341694,12.965590,87.034410,145.218249,4.987763,6.655453,8.956287,4.701747,104.073253,...,3.172085,3.688033,1.964977,11.403789,22.969206,9.953627,9.751733,19.034497,2544.064522,5.867807
std,5.087730e+04,9.626256,6.754526,6.754526,154.823626,3.926121,20.732450,12.557807,4.699044,129.075563,...,3.050455,3.521125,2.048710,45.290140,59.944724,19.463239,13.378119,65.556642,43704.613438,13.561230
min,-1.123052e+05,-258.947223,-0.264501,58.317865,21.295925,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,-73.171448,-76.131089,-100.000000,-88.470411,-68.455858,-100.000000,-77.253164
25%,8.840359e+01,7.990934,8.570496,84.172115,105.933679,2.847946,0.000000,2.443350,2.353552,58.415206,...,1.640437,0.992060,0.436808,-12.467466,-8.642882,3.420324,2.991587,-3.045542,3.048667,1.397009
50%,2.636429e+02,9.205931,11.567498,88.432502,114.004995,4.608605,1.470285,5.271447,3.453965,81.433699,...,2.416134,2.623152,1.329306,4.167747,13.778457,9.801012,9.117156,7.058723,12.253199,6.361039
75%,5.604946e+02,14.261135,15.827885,91.429504,130.653288,6.293386,2.841323,11.245163,5.420679,105.891938,...,3.567129,5.464191,2.751399,23.687607,36.864567,15.562938,14.334863,22.425626,23.123778,10.347918
max,1.760600e+06,34.196266,41.682135,100.264501,2350.920665,43.076268,100.000000,100.000000,31.307321,1465.731674,...,20.502539,18.981181,11.616699,781.027486,472.451991,106.701001,84.916787,864.953131,949405.933333,104.889489


In [27]:
dataset_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 1208 entries, 96 to 1303
Data columns (total 34 columns):
 #   Column                                                             Non-Null Count  Dtype  
---  ------                                                             --------------  -----  
 0   MES                                                                1208 non-null   str    
 1   AÑO                                                                1208 non-null   str    
 2   ENTIDAD                                                            1208 non-null   str    
 3   ( PATRIMONIO + RESULTADOS ) / ACTIVOS INMOVILIZADOS NETOS (3) (6)  1208 non-null   float64
 4   INDICE DE CAPITALIZACION NETO: FK / FI                             1208 non-null   float64
 5   ACTIVOS IMPRODUCTIVOS NETOS / TOTAL ACTIVOS                        1208 non-null   float64
 6   ACTIVOS PRODUCTIVOS / TOTAL ACTIVOS                                1208 non-null   float64
 7   ACTIVOS PRODUCTIVOS / PASIVOS CON 

In [28]:
# Se construye una tabla de diagnóstico de valores nulos.
#
# Cantidad_nulos indica cuántas observaciones faltantes existen
# en cada variable.
#
# Porcentaje_nulos expresa los valores faltantes como porcentaje
# del total de observaciones.

nulos = pd.DataFrame({
    "Variable": dataset_final.columns,
    "Cantidad_nulos": dataset_final.isna().sum().values,
    "Porcentaje_nulos": (
        dataset_final.isna().mean().values * 100
    )
})

nulos

,Variable,Cantidad_nulos,Porcentaje_nulos
0,MES,0,0.000000
1,AÑO,0,0.000000
2,ENTIDAD,0,0.000000
3,( PATRIMONIO + RESULTADOS ) / ACTIVOS INMOVILI...,0,0.000000
4,INDICE DE CAPITALIZACION NETO: FK / FI,0,0.000000
5,ACTIVOS IMPRODUCTIVOS NETOS / TOTAL ACTIVOS,0,0.000000
6,ACTIVOS PRODUCTIVOS / TOTAL ACTIVOS,0,0.000000
7,ACTIVOS PRODUCTIVOS / PASIVOS CON COSTO,0,0.000000
8,MOROSIDAD DE LA CARTERA DE CREDITOS CONSUMO,0,0.000000
9,MOROSIDAD DE LA CARTERA DE CRÉDITOS INMOBILIARIO,0,0.000000


In [29]:
# Se ordenan las variables de mayor a menor cantidad de valores
# nulos para identificar rápidamente aquellas que requieren
# revisión.


nulos = nulos.sort_values(
    "Cantidad_nulos",
    ascending=False
).reset_index(drop=True)

nulos

,Variable,Cantidad_nulos,Porcentaje_nulos
0,CARTERA VENCIDA / CARTERA BRUTA,1,0.082781
1,CARTERA IMPRODUCTIVA / CARTERA BRUTA,1,0.082781
2,CARTERA REFINANCIADA / CARTERA BRUTA,1,0.082781
3,CARTERA REESTRUCTURADA / CARTERA BRUTA,1,0.082781
4,CARTERA QUE NO DEVENGA INTERESES / CARTERA BRUTA,1,0.082781
5,MES,0,0.000000
6,AÑO,0,0.000000
7,ENTIDAD,0,0.000000
8,INDICE DE CAPITALIZACION NETO: FK / FI,0,0.000000
9,( PATRIMONIO + RESULTADOS ) / ACTIVOS INMOVILI...,0,0.000000


Se identifican los indicadores de cartera que presentaron
observaciones nulas durante la validación.

Estas variables se construyen a partir de divisiones entre
componentes de cartera y CARTERA BRUTA, por lo que un NaN
puede aparecer cuando el denominador no permite realizar
el cálculo.

In [30]:
columnas_con_un_nulo=["CARTERA QUE NO DEVENGA INTERESES / CARTERA BRUTA"
,"CARTERA REFINANCIADA / CARTERA BRUTA"
,"CARTERA VENCIDA / CARTERA BRUTA"
,"CARTERA IMPRODUCTIVA / CARTERA BRUTA"
,"CARTERA REESTRUCTURADA / CARTERA BRUTA"]


Se encontró un registro con valor `NAN` en todos las variables construidas de cartera. Esta  observación específica identificada corresponde a Marzo 2025 BP FINCA S.A./BANCO AMIBANK S.A. y se elimina este registro del dataset porque después de varias intervenciones por parte de la Superintendencia de Bancos, la entidad es cerrada e ingresa en un proceso de liquidación  el 11 de Marzo de 2025  debido a pérdidas continuas y deficiencias en su gestión.


In [31]:
dataset_final.loc[
    dataset_final[columnas_con_un_nulo].isna().any(axis=1),
    ["AÑO", "MES", "ENTIDAD"] + columnas_con_un_nulo
]

,AÑO,MES,ENTIDAD,CARTERA QUE NO DEVENGA INTERESES / CARTERA BRUTA,CARTERA REFINANCIADA / CARTERA BRUTA,CARTERA VENCIDA / CARTERA BRUTA,CARTERA IMPRODUCTIVA / CARTERA BRUTA,CARTERA REESTRUCTURADA / CARTERA BRUTA
931,2025,Marzo,BP FINCA S.A./BANCO AMIBANK S.A.,NaN,NaN,NaN,NaN,NaN


In [32]:
base_sin_eliminar=dataset_final.copy()

In [33]:
dataset_final.shape

(1208, 34)

In [34]:
dataset_final = dataset_final[
    ~(
        (dataset_final["AÑO"] == "2025") &
        (dataset_final["MES"].str.strip() == "Marzo") &
        (dataset_final["ENTIDAD"] == "BP FINCA S.A./BANCO AMIBANK S.A.")
    )
].copy()

In [35]:
dataset_final.shape

(1207, 34)

### Detección de duplicados

In [41]:
# Se verifica si existen registros repetidos para una misma
# entidad y período.
feature_engineering.detectar_duplicados(dataset_final)

np.int64(0)

### Detección de infinitos en las variables numéricas

In [42]:
feature_engineering.detectar_infinitos(dataset_final)

( PATRIMONIO + RESULTADOS ) / ACTIVOS INMOVILIZADOS NETOS (3) (6)    0
INDICE DE CAPITALIZACION NETO: FK / FI                               0
ACTIVOS IMPRODUCTIVOS NETOS / TOTAL ACTIVOS                          0
ACTIVOS PRODUCTIVOS / TOTAL ACTIVOS                                  0
ACTIVOS PRODUCTIVOS / PASIVOS CON COSTO                              0
MOROSIDAD DE LA CARTERA DE CREDITOS CONSUMO                          0
MOROSIDAD DE LA CARTERA DE CRÉDITOS INMOBILIARIO                     0
MOROSIDAD DE LA CARTERA DE CRÉDITOS MICROCRÉDITO                     0
MOROSIDAD DE LA CARTERA TOTAL                                        0
COBERTURA DE LA CARTERA REFINANCIADA                                 0
COBERTURA DE LA CARTERA REESTRUCTURADA                               0
COBERTURA DE LA CARTERA PROBLEMÁTICA                                 0
GASTOS DE OPERACION ESTIMADOS / TOTAL ACTIVO PROMEDIO (3)            0
GASTOS DE OPERACION  / MARGEN FINANCIERO                             0
GASTOS

### Entidades consideradas

In [43]:
dataset_final["ENTIDAD"].unique()

<StringArray>
[                                           'BP GUAYAQUIL',
                                             'BP PACIFICO',
                                            'BP PICHINCHA',
                                           'BP PRODUBANCO',
                                               'BP AUSTRO',
                                          'BP BOLIVARIANO',
                                             'BP CITIBANK',
                                               'BP DINERS',
                                    'BP GENERAL RUMIÑAHUI',
                                        'BP INTERNACIONAL',
                                                 'BP LOJA',
                                              'BP MACHALA',
                                            'BP SOLIDARIO',
                                            'BP PROCREDIT',
                                             'BP AMAZONAS',
                            'BP BANCO COMERCIAL DE MANABI',
                          

## Generación de base limpia y registro del preprocesamiento

In [44]:
# Se identifica la observación que será excluida de la base
# debido a los valores nulos detectados en los indicadores de cartera.
condicion_eliminacion = (
    (base_sin_eliminar["AÑO"] == "2025") &
    (base_sin_eliminar["MES"].str.strip() == "Marzo") &
    (base_sin_eliminar["ENTIDAD"] == "BP FINCA S.A./BANCO AMIBANK S.A.")
)

# Se obtiene la información de la fila antes de eliminarla,
# para conservar evidencia de la observación excluida.
fila_eliminada = base_sin_eliminar.loc[
    condicion_eliminacion
].copy()


variables_con_nulos = [
    columna
    for columna in columnas_con_un_nulo
    if fila_eliminada[columna].isna().any()
]



# --------------------------------------------------------
# Creación del log de preprocesamiento
# --------------------------------------------------------

log_preprocesamiento = {
    "proceso": "Preprocesamiento de datos",
    "fecha_ejecucion": datetime.now().isoformat(),
    "eliminaciones": [
        {
            "motivo": (
                "Se eliminó una observación debido a valores nulos "
                "detectados en indicadores de cartera."
                "debido a que la entidad entró en proceso de cierre y liquidación"
                "el 11 de Marzo de 2025"
            ),
            "anio": "2025",
            "mes": "Marzo",
            "entidad": "BP FINCA S.A./BANCO AMIBANK S.A.",
            "variables_con_nulos": variables_con_nulos,
            "filas_eliminadas": len(fila_eliminada),
            "tamaño del dataset":dataset_final.shape

        }
    ]
}

# --------------------------------------------------------
# Guardar el log en la carpeta results
# --------------------------------------------------------

ruta_results = RAIZ_PROYECTO / "results"

# Se crea la carpeta results si no existe.
ruta_results.mkdir(
    parents=True,
    exist_ok=True
)

ruta_log = (
    RAIZ_PROYECTO
    / "results"
    / "preprocessing_log1.json"
)

# Se guarda el registro en formato JSON.
with open(
    ruta_log,
    "w",
    encoding="utf-8"
) as archivo:

    json.dump(
        log_preprocesamiento,
        archivo,
        indent=4,
        ensure_ascii=False
    )

print(f"Log guardado en: {ruta_log}")


Log guardado en: d:\OneDrive - SUPERINTENDENCIA DE BANCOS\Escritorio\Proyecto_usfq\results\preprocessing_log1.json


Se guarda en un log el detalle de los resultados y en data_processed la base preprocesada.

In [45]:
## Se almacena una copia del dataset finalizado
base_final = dataset_final.copy()

In [46]:
# Se construye la ruta de salida a partir de la raíz del proyecto.
ruta_salida = RAIZ_PROYECTO / "data_processed" / "base_preprocesada1.xlsx"

# Se crea la carpeta de salida si aún no existe.
ruta_salida.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Se guarda la base preprocesada.
base_final.to_excel(
    ruta_salida,
    index=False
)

print(f"Base preprocesada guardada en: {ruta_salida}")

Base preprocesada guardada en: d:\OneDrive - SUPERINTENDENCIA DE BANCOS\Escritorio\Proyecto_usfq\data_processed\base_preprocesada1.xlsx


### Conclusiones:

1. Se logró integrar información de diferentes fuentes y años.
El flujo integra los indicadores, información de cartera y variaciones del balance para el período 2022–2026. Esto deja una base estructurada por AÑO, MES y ENTIDAD, que posteriormente permite comparar el comportamiento de cada banco a través del tiempo.
2. La información quedó estandarizada en variables numéricas, y se multiplicó por 100.
3. En el preprocessing conviertes las variables financieras a float64 y las variables categóricas (MES, AÑO, ENTIDAD) a texto. Esto es importante porque las fuentes originales son archivos Excel y pueden contener valores almacenados como texto. Además, en el feature engineering también conviertes explícitamente las variables financieras a numéricas usando errors="coerce".
Se identificaron y trataron valores nulos derivados de los indicadores de cartera.
Las variables de cartera se construyen dividiendo cada componente para CARTERA BRUTA. Cuando la cartera bruta es 0, el código reemplaza ese denominador por NaN, evitando realizar una división entre cero.
Por tanto, los nulos encontrados en esas variables no necesariamente representan datos faltantes originales; pueden ser consecuencia de que no existe un denominador válido para calcular el indicador.
4. Se detectó un registro que debía excluirse del análisis.
En el notebook identificas específicamente el registro correspondiente a marzo de 2025 de BP FINCA S.A./BANCO AMIBANK S.A. y posteriormente lo eliminas, esto debido a que el 11 de Marzo de 2025 la entidad entró en proceso de liquidación, razón de la inexistencia de datos de cartera.
5. No existen duplicados utilizando la llave lógica del dataset: AÑO, MES y ENTIDAD como identificadores. Esto es adecuado porque, conceptualmente, debería existir una observación por banco para cada período mensual.
6. Se verificó la presencia de valores infinitos.
Incluyes explícitamente una revisión de inf y -inf en las variables numéricas. Esto es especialmente importante porque varias variables son ratios y podrían generar infinitos si un denominador fuera cero. No existen valores infinitos.
7. Se analizarán un total de 24 entidades financieras privadas desde Mayo 2021 a Julio 2026, período en el que el ente de control realizó un proceso de homologación de la información financiera para que los datos sean comparables en el tiempo

In [49]:
base_final

,MES,AÑO,ENTIDAD,( PATRIMONIO + RESULTADOS ) / ACTIVOS INMOVILIZADOS NETOS (3) (6),INDICE DE CAPITALIZACION NETO: FK / FI,ACTIVOS IMPRODUCTIVOS NETOS / TOTAL ACTIVOS,ACTIVOS PRODUCTIVOS / TOTAL ACTIVOS,ACTIVOS PRODUCTIVOS / PASIVOS CON COSTO,MOROSIDAD DE LA CARTERA DE CREDITOS CONSUMO,MOROSIDAD DE LA CARTERA DE CRÉDITOS INMOBILIARIO,...,CARTERA QUE NO DEVENGA INTERESES / CARTERA BRUTA,CARTERA REFINANCIADA / CARTERA BRUTA,CARTERA REESTRUCTURADA / CARTERA BRUTA,VAR_FONDOS DISPONIBLES,VAR_INVERSIONES,VAR_CARTERA DE CRÉDITOS,VAR_TOTAL ACTIVO,VAR_Depósitos a la vista,VAR_Depósitos a plazo,VAR_TOTAL PATRIMONIO
96,Mayo,2022,BP GUAYAQUIL,410.634510,8.120787,13.346535,86.653465,120.426351,1.549652,1.820555,...,0.704908,1.710131,0.527229,-15.357506,-3.105869,21.963595,10.850009,14.419397,16.334925,8.087612
97,Mayo,2022,BP PACIFICO,276.006994,9.549144,21.715553,78.284447,111.274525,3.261802,4.401917,...,1.798705,9.543764,4.166759,-16.102362,15.214273,-0.671445,-0.215742,-3.749551,7.616585,2.003489
98,Mayo,2022,BP PICHINCHA,-1288.206946,8.728095,8.727610,91.272390,142.387866,2.101977,4.277935,...,1.743190,2.169764,4.722154,-34.596167,0.889363,34.189556,10.684144,9.095698,17.660489,4.376836
99,Mayo,2022,BP PRODUBANCO,345.788698,7.046035,11.941164,88.058836,146.171009,2.909608,2.891999,...,1.062750,0.607335,2.484534,-4.501507,-3.521776,24.150188,13.806515,9.297570,11.956282,7.797315
100,Mayo,2022,BP AUSTRO,129.543127,7.715342,16.033362,83.966638,104.258095,7.746817,1.672858,...,2.521220,7.008823,2.783564,-15.937829,6.911905,21.409839,11.400689,7.823076,4.820617,12.813348
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1299,Julio,2026,BP CAPITAL,74.360823,8.877858,20.311775,79.688225,97.499072,3.526914,0.000000,...,4.118464,1.431514,0.247529,-1.764260,54.532954,43.405379,32.243631,4.628479,44.564605,4.072887
1300,Julio,2026,BP DELBANK,743.520441,28.876492,15.797866,84.202134,176.981056,10.591267,0.000152,...,2.017141,6.891239,0.384947,9.622450,-37.925514,-11.897364,-10.244491,-12.542475,-22.706797,5.183978
1301,Julio,2026,BP D-MIRO S.A./BANCO ATLÁNTIDA S.A.,78.836924,5.519091,12.819096,87.180904,96.598989,0.035790,0.000000,...,1.235770,2.883851,0.751003,139.889649,50.905153,78.103791,71.293760,27.512148,109.688042,-5.648516
1302,Julio,2026,"BP BANCO DESARROLLO DE LOS PUEBLOS S.A., COD...",299.062134,10.365757,10.524893,89.475107,106.919132,7.961921,0.608309,...,5.890426,4.113364,0.662222,-58.597055,24.865624,3.694559,-5.187257,-13.838419,8.657541,6.703112


In [48]:
base_final["ENTIDAD"].unique()

<StringArray>
[                                           'BP GUAYAQUIL',
                                             'BP PACIFICO',
                                            'BP PICHINCHA',
                                           'BP PRODUBANCO',
                                               'BP AUSTRO',
                                          'BP BOLIVARIANO',
                                             'BP CITIBANK',
                                               'BP DINERS',
                                    'BP GENERAL RUMIÑAHUI',
                                        'BP INTERNACIONAL',
                                                 'BP LOJA',
                                              'BP MACHALA',
                                            'BP SOLIDARIO',
                                            'BP PROCREDIT',
                                             'BP AMAZONAS',
                            'BP BANCO COMERCIAL DE MANABI',
                          